# Step 1: PDF ingestion (27-5-26)

## pdf_extractor.py

### A. text, table, image extraction with pdfplumber

In [ ]:
# pip show pillow - to save images into .png (It's the standard Python image library. Lightweight)

In [ ]:
import pdfplumber
import os
from langchain_core.documents import Document
from PIL import Image
from io import BytesIO
import shutil
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
DATA_FOLDER = "C:/Users/Pranali Jadhav/OneDrive/Documents/GEN_AI/my_study/Bot_Project_1/graphprod/"

def loader_doc():
    text_doc = []
    table_doc =[]
    image_doc = []

    for file in os.listdir(DATA_FOLDER):
        if file.endswith(".pdf"):
            file_name = file.replace(".pdf", "")
            with pdfplumber.open(f"{DATA_FOLDER}/{file}") as pdf:
                
                if os.path.exists("images"):
                    shutil.rmtree("images")                
                os.makedirs("images", exist_ok=True)
                for page in pdf.pages:
                    if not page:
                        continue
                    else:
                        texts = page.extract_text()
                        
                        text_doc.append(Document(
                                page_content=texts,
                                    metadata={
                                        "page_number": page.page_number,
                                        "section": f"page_{page.page_number}",
                                        "chunk_type": "text",
                                        "source_file": file
                                        # "image_path": {image_path} # image only
                                        }
                                    ))
                        tables = page.extract_tables()
                        for i, table in enumerate(tables):
                            table_doc.append(Document(
                                page_content=str(table),
                                    metadata={
                                        "page_number": page.page_number,
                                        "section": f"page_{page.page_number}",
                                        "chunk_type": "table",
                                        "source_file": file
                                        # "image_path": {image_path} # image only
                                        }
                                    ))
                        images = [img for img in page.images if img["srcsize"][0] >= 100 and img["srcsize"][1] >= 100]                        
                        for i, img in enumerate(images):
                            try:
                                raw_bytes = img["stream"].get_data()
                                pil_image = Image.open(BytesIO(raw_bytes))
                                if pil_image.mode == "CMYK":
                                    pil_image = pil_image.convert("RGB")
                                image_path = pil_image.save(f"images/page_{page.page_number}_image_{i}.png")
                                image_doc.append(Document(
                                    page_content=f"images/page_{page.page_number}_image_{i}.png",
                                    metadata={
                                        "page_number": page.page_number,
                                        "section": f"page_{page.page_number}",
                                        "chunk_type": "image_description",
                                        "source_file": file,
                                        "image_path": image_path # image for only
                                        }
                                    ))
                            except Exception as e:
                                print(f"Skipped image on page {page.page_number}: {e}")
                                continue

    # print(len(text_doc))
    # print(len(table_doc))
    # print(len(image_doc))
    return text_doc, table_doc, image_doc
                        


In [ ]:
# loader_doc() #-- testing

In [ ]:
text_doc, table_doc, image_doc = loader_doc()
# print(len(text_doc))

In [ ]:
# print(type(text_doc[0]))
# print(text_doc[0])

### B. Chunking (Chunker.py)

In [ ]:
def chunker(text_doc):
    splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
    chunks = splitter.split_documents(text_doc)
    len(chunks)
    print(chunks[0].page_content)
    print(chunks[0].metadata)
    return chunks

In [ ]:
# chunker(text_doc)

chunks = chunker(text_doc)
print(len(chunks))

In [ ]:
table_chunks = chunker(table_doc)
print(len(table_chunks))

### C. image_processor (image_processor.py)- will get context of the mage that we extracted from  the pdf.

In [ ]:
import base64
from openai import OpenAI
import pickle

In [ ]:
def image_processor(image_doc):
    client = OpenAI()
    for i, img in enumerate(image_doc):
        try:
                with open(img.page_content, "rb") as f:
                    image_data = base64.b64encode(f.read()).decode("utf-8")              

                response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/png;base64,{image_data}"
                                }
                            },
                            {
                                "type": "text",
                                "text": """You are analyzing a BMW service manual image, be specific and technical so the description is useful for a technician 
                                searching for information. look for any numbers, labels and text, what that images shows, check for symbols and indicators."""
                            }
                        ]
                    }
                ]
                )
            
                description = response.choices[0].message.content
                img.page_content = description
                print(f"Processing {i+1}/{len(image_doc)}: {img.metadata['page_number']}") #  progress counter inside the loop
        except Exception as e:
                print(f"Skipped {img.page_content}: {e}")
                continue
    
    with open("processed_images.pkl", "wb") as f:
            pickle.dump(image_doc, f)
    return image_doc

In [ ]:
# testing
# processed_images = image_processor(image_doc[10:13])
processed_images = image_processor(image_doc)
# print(processed_images[0].page_content)
# print(len(processed_images))

In [ ]:
# testing the file description got from image_processor which we have saved loacally so we dont have to call the LLM again for all 375 images its costly.
with open("processed_images.pkl", "rb") as f:
    processed_images = pickle.load(f)

print(len(processed_images))
print(processed_images[10].page_content)

In [ ]:
# count how many still have file paths (skipped)
skipped = [img for img in processed_images if img.page_content.startswith("images/")]
print(f"Skipped: {len(skipped)}")
print(f"Processed: {len(processed_images) - len(skipped)}")

In [ ]:
# testing
# print(processed_images[50].page_content)
# print(processed_images[50].metadata)

In [ ]:
#check if chunking needed for the image description. But Most image descriptions are short — 100-300 words. Well under your 2000 character chunk size.So in most cases no chunking needed.
image_chunks = chunker(processed_images)
print(len(image_chunks)) # lenth check

### D. Vectore_store (Vectore_store.py) : storing all embadded vectors in VectorDB

In [ ]:
import chromadb
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
def vector_store(chunks, collection_name, persist_dir='./BMW_RAG_db'):
    embedding_model=OpenAIEmbeddings(model='text-embedding-3-small')
    vector_store=Chroma.from_documents(
        documents=chunks,
        collection_name=collection_name,
        embedding=embedding_model,
        persist_directory=persist_dir,
        collection_metadata={"hnsw:space":'cosine'}
    )

    print(f"Collection '{collection_name}' created/updated with {len(chunks)} chunks.")

    return vector_store

In [ ]:
# we send chunks to vectore store from here then storing them back into variable to pass them further
table_store = vector_store(table_chunks, "table_chunks")
text_store = vector_store(chunks + image_chunks, "text_chunks")

In [ ]:
# **** this is temparary to use vectdb vectorDB locally instade recreate thenm everytime of the run. **** 

import chromadb
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

client_db = chromadb.PersistentClient(path="./BMW_RAG_db")

text_store = Chroma(
    client=client_db,
    collection_name="text_chunks",
    embedding_function=embedding_model
)

table_store = Chroma(
    client=client_db,
    collection_name="table_chunks",
    embedding_function=embedding_model
)

In [ ]:
# **** this is temparary to use vectdb vectorDB locally instade recreate thenm everytime of the run. **** 
print(text_store._collection.count())   # should print 1101
print(table_store._collection.count())  # should print 1

In [ ]:
# testing -
# 1.
# results = text_store.similarity_search("parking brake", k=3)
# for r in results:
#     print(r.page_content)
#     print(r.metadata)
#     print("---")
# # 2.
# results = text_store.similarity_search("steering wheel controls", k=3)
# for r in results:
#     print(r.page_content[:200])
#     print(r.metadata)
#     print("---")

# ======== Step 1 complete ===============================

# Step 2 - Query classifier (LangGraph node : Date - 29-2026)

## A. state.py

#### IMP ---- we have consider the context in architecture but not using them in project as we have only 1 manual
####(Vehicle context = (make, model, year, engine))

In [ ]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
import operator
from openai import OpenAI

In [ ]:
class Citation(TypedDict): # this class is needed because the agent was extracting and tracking citations per retrieved chunk, and you needed a typed container for that.
    page: int # otherwise we dont need this class only Agentstate would be enough.
    section: str
    source: str

class AgentState(TypedDict):
    user_type: str
    query: str
    intent: str
    retrieved_chunks: Annotated[list[dict], operator.add]  # ✅ merge parallel writes
    confidence_score: float
    conversation_history: Annotated[list[BaseMessage], operator.add] 
    citations: Annotated[list[Citation], operator.add]  
    image_paths: Annotated[list[str], operator.add]  

### B. Classifier to classify from which vectoreDB need to find An answer for the user query?
    Tables and text are structurally different. If someone asks: -
    "what does the oil warning light mean" → answer is in prose text, search text_chunks
    "what is the torque spec for front wheel bolts" → answer is in a table row, search table_chunks
    2. What intent means
    Intent = what the user is trying to do, not the literal words they used.
    Same underlying need, different phrasings:
    "there's a yellow circle on my dash"
    "warning light came on"              →  intent for all three is : warning_light
    "my car is showing some symbol"

### classifier.py

In [ ]:
client = OpenAI()
def classifier (state):
    VALID_ROUTES = ["text", "table", "both", "unknown"]
    # added enriched_history for follow-up questions
    history = state["conversation_history"]
    if history:
    # get last 2 messages
        recent = history[-2:] if len(history) >= 2 else history
        context = "\n".join([
            f"{'User' if isinstance(m, HumanMessage) else 'Assistant'}: {m.content}"
            for m in recent
        ])
        enriched_query = f"Previous exchange:\n{context}\n\nCurrent query: {state['query']}"
    else:
        enriched_query = state["query"]

    response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                {"role": "system", "content": """You are a routing assistant for a BMW service manual.
                        Given a user query, decide which data source to search.

                        Return only one of-
                        text: descriptive information, procedures, explanations, warnings, 
                        fluid types and specifications (e.g. which oil to use), 
                        component descriptions, operating instructions
                 
                        table: numerical specs, exact measurements, torque values (e.g. 25 Nm), 
                        service intervals (e.g. every 10,000 miles), scheduled maintenance dates, 
                        fault/error codes, fluid capacities in exact quantities
  
                        both: multiple symptoms, diagnostic queries, anything where 
                        the answer needs both an explanation AND a spec value, 
                        strange noises, car not starting, complex issues
                 
                        unknown: ONLY use this for queries that are completely 
                        outside vehicle service manual scope — prices, dealer 
                        locations, purchase advice, non-automotive topics.
                        Any query about vehicle symptoms, warnings, maintenance, 
                        repairs, or parts belongs to text/table/both."""},
                {"role": "user", "content": enriched_query}
                ])
    
    intent = response.choices[0].message.content.strip().lower()
    if intent not in VALID_ROUTES:
        intent = "unknown"
    return {"intent": intent}
                    

In [ ]:
#testing after enriched the query.
state = {
    "query": "Is this something I can fix myself?",
    "user_type": "owner",
    "conversation_history": [
        HumanMessage(content="My BMW is making a strange noise when I brake"),
        AIMessage(content="A strange noise when braking could be a sign of various issues...")
    ],
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "citations": [],
    "image_paths": []
}

result = classifier(state)
print(result)

In [ ]:
# testing :
# Test queries
queries = [
    "there is a yellow circle on my dashboard",
    "what is the torque spec for front wheel bolts",
    "when is my next oil change due",
    "my car is making a strange noise and won't start",
    "tell me oprating element on steering wheel",
    "tell me price of head light"
]

for q in queries:
    state = {"query": q}
    result = classifier(state)
    print(f"Query: {q}")
    print(f"Intent: {result['intent']}")
    print("---")

In [ ]:
# 1. testing
test_state = {
    "user_type": "owner",
    "query": "there is a yellow warning light on my dashboard",
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = app.invoke(test_state)
print("Intent:", result["intent"])

In [ ]:
# 2. Now test the diagnostic case — this is the one that triggers parallel execution of both retrievers:
test_state = {
    "user_type": "technician",
    "query": "my car is making a strange noise and won't start",
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = app.invoke(test_state)
print("Intent:", result["intent"])

In [ ]:
from langgraph.graph import StateGraph, END

def route_intent(state):
    intent = state["intent"]
    if intent == "both":
        return ["text_retriever", "table_retriever"]
    elif intent == "table":
        return "table_retriever"
    elif intent == "unknown":
        return "unknown_handler"
    else:
        return "text_retriever"

### empty chunks handler node

In [ ]:
def unknown_handler(state):
    return {
        "conversation_history": state["conversation_history"] + [
            AIMessage(content="I couldn't find relevant information in the BMW service manual for your query. Please consult a certified BMW technician or visit your nearest service center.")
        ]
    }

# ======== Step 2 complete ===============================

# Step 3 - Create retivers/empty chunk handling (30-5-26)

## text_retriever.py

In [ ]:
def text_retriever(state):
    results = text_store.similarity_search_with_score(state["query"], k=5)
    chunks = []  
    for doc, score in results:
        chunks.append({ 
            "content": doc.page_content,
            "metadata": doc.metadata,
            "score": score
        })
    return {"retrieved_chunks": chunks}

In [ ]:
# testing 
test_state = {
    "user_type": "owner",
    "query": "what does the oil warning light mean",
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = app.invoke(test_state)
print("Intent:", result["intent"])
print("Chunks retrieved:", len(result["retrieved_chunks"]))
for chunk in result["retrieved_chunks"]:
    print("Score:", chunk["score"])
    print("Content preview:", chunk["content"][:100])
    print("---")

### table_retriever.py

In [ ]:
def table_retriever(state):
    results = table_store.similarity_search_with_score(state["query"], k=5)
    chunks = []  
    for doc, score in results:
        chunks.append({ 
            "content": doc.page_content,
            "metadata": doc.metadata,
            "score": score
        })
    return {"retrieved_chunks": chunks}

In [ ]:
# testing
test_state = {
    "user_type": "technician",
    "query": "what is the torque spec for front wheel bolts",
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = app.invoke(test_state)
print("Intent:", result["intent"])
print("Chunks retrieved:", len(result["retrieved_chunks"]))
for chunk in result["retrieved_chunks"]:
    print("Score:", chunk["score"])
    print("Content preview:", chunk["content"][:100])
    print("---")

In [ ]:
# testing
queries = [
    ("there is a yellow circle on my dashboard", "text route"),
    ("what is the torque spec for front wheel bolts", "table route"),
    ("my car is making a strange noise and won't start", "both route"),
    ("tell me the price of a headlight", "unknown route")
]

for query, expected in queries:
    result = app.invoke({
        "user_type": "owner",
        "query": query,
        "intent": "",
        "retrieved_chunks": [],
        "confidence_score": 0.0,
        "conversation_history": [],
        "citations": [],
        "image_paths": []
    })
    print(f"Query: {query}")
    print(f"Expected: {expected}")
    print(f"Intent: {result['intent']}")
    print(f"Chunks: {len(result['retrieved_chunks'])}")
    print("---")

# ======== Step 3 complete ===============================

# Step 4 : Multi-turn diagnostic conversation + memory, connfidence and citation

## A. conversation.py

In [ ]:
# from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
# import operator
from openai import OpenAI

In [ ]:
client = OpenAI()

def conversation(state):
    if not state["retrieved_chunks"]:
        return {
            "conversation_history": state["conversation_history"] + [
                AIMessage(content="I couldn't find relevant information in your BMW manual for this query. Please consult a certified BMW technician.")
            ]
        }

    context = "\n\n".join([
        f"Page {chunk['metadata'].get('page_number', 'unknown')}:\n{chunk['content']}"
        for chunk in state["retrieved_chunks"]
    ])

    user_type = state["user_type"]
    citation_text = "\n".join([
    f"- Page {c['page']}, Section {c['section']}"
    for c in state["citations"]])

    if user_type == "owner":
        system_prompt = f"""You are a BMW service manual assistant helping a car owner.
                        Use simple, non-technical language. Avoid jargon.
                        Always recommend visiting a certified BMW service center for repairs.
                        Base your answer only on the provided manual context.
                        Reference these manual sections: {citation_text}
                        Keep the response concise and under 150 words."""

    else:
        system_prompt = f"""You are a BMW service manual assistant helping a certified technician.
                        Use precise technical language. Include specifications, torque values, and part references where available.
                        Always cite the page number from the manual context in your response.
                        Base your answer only on the provided manual context.
                        Reference these manual sections: {citation_text}"""
                        

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            *state["conversation_history"],
			
            {"role": "user", "content": f"{state['query']}\n\nContext:\n{context}"}
        ]
    )

    answer = response.choices[0].message.content.strip()
    
    if state["confidence_score"] > 0.7:
        answer += "\n\n⚠️ Note: This answer is based on limited matches from the manual. Please verify with a certified BMW technician."

    return {
        "conversation_history": state["conversation_history"] + [
            HumanMessage(content=state["query"]),
            AIMessage(content=answer)
        ]
    }

In [ ]:
# testing 1 - an empty chunks
test_state = {
    "user_type": "owner",
    "query": "What is the recommended oil for my BMW?",
    "intent": "text",
    "retrieved_chunks": [[(Document(id='ae2473c1-d090-4e0b-92f9-676ea1e7a1bf', metadata={'chunk_type': 'text', 'source_file': 'bmw_manual.pdf', 'page_number': 417, 'section': 'page_417'}, page_content='Seite 417\nOperating fluids MOBILITY\nPetrol engine\nNOTICE\nACEA C2.\nUsing the wrong engine oil can result in en-\ngine malfunctions and damage. There is a ACEA C3.\nrisk of material damage. When selecting the\nACEA C5.\nengine oil, make sure that it is the correct oil\nspecification.\nDiesel engine\nSuitable engine oil grades ACEA C2.\nEngine oil with the following oil specification ACEA C3.\ncan be topped up:\nPetrol engine with exhaust gas particulate Viscosity classes\nfilter When selecting an engine oil, make sure that\nthe engine oil belongs to one of the following\nBMW Longlife-12 FE.\nviscosity classes:\nBMW Longlife-17 FE+.\nPetrol engine\nBMW Longlife-19 FE.\nSAE 0W-12.\nBMW Longlife-22 FE++.\nSAE 0W-20.\nSAE 0W-30.\nPetrol engine without exhaust gas particu-\nlate filter\nThe viscosity classes SAE 0W-12 and SAE\nBMW Longlife-01 FE. 0W-20 are not suitable for the 60i petrol en-\ngine.\nBMW Longlife-17 FE+.\nDiesel engine\nBMW Longlife-22 FE++.\nSAE 0W-30.\nThe oil specifications BMW Longlife-17 FE+\nand BMW Longlife-22 FE+ are not suitable for Viscosity classes with a high viscosity grade\nuse with 60i petrol engines. can increase fuel consumption.\nDiesel engine Further information on suitable engine oil\nspecifications and viscosity classes can be ob-\nBMW Longlife-12 FE.\ntained from an authorised Service Partner or\nBMW Longlife-19 FE. another qualified Service Partner or a specialist\nworkshop.\nAlternative engine oil grades\nIf suitable engine oils are not available, up to\n1 litre, 2 pints of an engine oil with the following\noil specification can be used for topping up:\n417\nOnline Edition for Part no. 01405B64643 - II/25'), 0.3853793144226074), (Document(id='57053802-4d79-4e47-b9fb-2eda0fc74d98', metadata={'page_number': 34, 'section': 'page_34', 'chunk_type': 'image_description', 'source_file': 'bmw_manual.pdf'}, page_content='The image shows an engine oil filler cap, commonly found in a BMW vehicle. The cap has an illustration of an oil can symbol, which indicates its purpose for adding engine oil. There is an arrow pointing counterclockwise, suggesting the direction to turn the cap to open it. No numbers, text, or additional labels are visible aside from the symbols and arrow. This cap is typically located on the top of the engine and is a crucial component for maintaining proper engine lubrication by allowing oil to be added easily.'), 0.4909687042236328), (Document(id='2dbefea3-ed87-408f-a24e-0e5456afc829', metadata={'source_file': 'bmw_manual.pdf', 'chunk_type': 'image_description', 'page_number': 77, 'section': 'page_77'}, page_content='The image shows a technical diagram of a component that appears to be an oil or fluid filter used in BMW vehicles. It is viewed from the top, with the primary focus on the circular filter element. The surrounding structure includes a housing that may be integrated into the engine or fluid system.\n\n1. **Central Component**: The dominant round element is likely the filter medium itself. It is enclosed by a circular housing which would be designed to fit into its designated slot within the vehicle.\n\n2. **Housing Details**: To one side, there is an L-shaped extension which is likely part of the filter housing. This part might include connection points or clips for securing the filter in place.\n\n3. **Seal and O-Rings**: The presence of a darker, possibly rubber ring around the circular component suggests a seal or an O-ring, crucial for preventing leaks.\n\n4. **Material Design**: The housing appears to be composed of light-colored material, potentially plastic or metal coated with a protective layer to withstand engine temperatures and fluid corrosion.\n\n5. **Possible Labels or Text**: There are no visible numbers or text on the housing visible in this particular angle of the diagram.\n\n6. **Connections and Ports**: The openings or tabs seen on the right side of the housing may indicate connection points for fluid lines or sensor integration.\n\nThis image would be useful for a technician when identifying the correct filter type and ensuring proper orientation during installation or replacement in a BMW vehicle.'), 0.5000331401824951), (Document(id='70fc082c-82cf-4bcd-8fda-3c02836adba3', metadata={'chunk_type': 'text', 'page_number': 5, 'section': 'page_5', 'source_file': 'bmw_manual.pdf'}, page_content='© 2025 Bayerische Motoren Werke\nAktiengesellschaft\nMunich, Germany\nNot to be reproduced, wholly or in part, without written permission from BMW AG, Munich.\nEnglish ID8 II/25, -\nPrinted on environmentally friendly paper, bleached without chlorine, suitable for recycling.\n5\nOnline Edition for Part no. 01405B64643 - II/25'), 0.5020538568496704), (Document(id='1de56dc8-eda4-486a-b5e2-6f9c561abf38', metadata={'page_number': 1, 'section': 'page_1', 'chunk_type': 'text', 'source_file': 'bmw_manual.pdf'}, page_content="Content A-Z\nOWNER'S HANDBOOK.\nBMW X5.\nOnline Edition for Part no. 01405B64643 - II/25"), 0.517418622970581)]],  # test empty path first
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = text_store.similarity_search_with_score("recommended oil BMW", k=5)
print(result)

In [ ]:
# testing 1 actual owners test case
raw = text_store.similarity_search_with_score("recommended oil BMW", k=5)

test_state = {
    "user_type": "owner",
    "query": "What is the recommended oil for my BMW?",
    "intent": "text",
    "retrieved_chunks": [
        {"content": doc.page_content, "metadata": doc.metadata, "score": score}
        for doc, score in raw
    ],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = conversation(test_state)
print(result)

# ======== Step 4 complete ===============================

# Step 5 : Confidance_score + citation 

## confidence.py

In [ ]:
def confidence_score (state):
    chunks = state["retrieved_chunks"]
    if not chunks:
        return {"confidence_score": 0.0, "citations": []}
    avg_score = sum(chunk["score"] for chunk in chunks) / len(chunks)
        
    citation = []
    for chunk in chunks:
        citation.append({
                "page": chunk["metadata"].get("page_number"),
                "section": chunk["metadata"].get("section"),
                "source": chunk["metadata"].get("source_file")
            })              
    return {"confidence_score" : avg_score, "citations" : citation}
   

In [ ]:
# testing
raw = text_store.similarity_search_with_score("recommended oil BMW", k=5)

test_state = {
    "user_type": "owner",
    "query": "What is the recommended oil for my BMW?",
    "intent": "text",
    "retrieved_chunks": [
        {"content": doc.page_content, "metadata": doc.metadata, "score": score}
        for doc, score in raw
    ],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
}

result = confidence_score(test_state)
print(result)

# ======== Step 5 complete ===============================

# Step 6 : AzureOpenAI() swap

#### For now, staying on OpenAI() is fine. The swap to AzureOpenAI() is a Step 6 task — just a client swap, not an architectural change.Want to skip Step 6 for now and move to Step 7

# ======== Step 6 complete ===============================

# Step 7: FastAPI + Docker 

## main.py

In [ ]:
import uvicorn
from api.app import api

if __name__ == "__main__":
    uvicorn.run(api, host="0.0.0.0", port=8000, reload=True)


## app.py

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from agent.graph import app as agent_app
from agent.state import Citation

api = FastAPI() # FastAPI() creates your web application instance.
sessions: dict = {}

# Request — what client sends
class QueryRequest(BaseModel):
    query: str
    session_id: str
    user_type: str

# Response — what you send back
class QueryResponse(BaseModel):
    answer: str
    citations: list[Citation]
    confidence_score: float

@api.post("/query")
def query(request: QueryRequest):
    result = agent_app.invoke({
    "query": request.query,
    "user_type": request.user_type,
    "conversation_history": sessions.get(request.session_id, []),  # loaded from sessions dict
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "citations": [],
    "image_paths": []
    })
    last_message = result["conversation_history"][-1].content #That line reads the last message from the result. It does not save anything, This READS the last AIMessage content to return as the answer
    sessions[request.session_id] = result["conversation_history"] #This SAVES the full updated history back into sessions
    return QueryResponse(
    answer=last_message,
    citations=result["citations"],
    confidence_score=result["confidence_score"]
)

# Graph = includes all steps as it executes the whole conversation loop

##### Graph - v1

In [ ]:

graph = StateGraph(AgentState)
graph.add_node('classifier', classifier)
graph.add_node('text_retriever', text_retriever)   
graph.add_node('table_retriever', table_retriever) 
graph.add_node('unknown_handler', unknown_handler)   
graph.add_node("confidence", confidence_score)
graph.add_node("conversation", conversation)

graph.set_entry_point('classifier')
graph.add_conditional_edges(
    "classifier",      # after this node
    route_intent,      # call this function
    {
        "text_retriever": "text_retriever",
        "table_retriever": "table_retriever",
        "unknown_handler" : "unknown_handler"
    }
)
# graph.add_edge('router', 'retriever') 
graph.add_edge('text_retriever', 'confidence')   
graph.add_edge('table_retriever', 'confidence')  
graph.add_edge('unknown_handler', END)
graph.add_edge('confidence', 'conversation') 
graph.add_edge('conversation', END) 


app = graph.compile()

In [ ]:
# testing - whole graph after adding confidence_score and Citation
result = app.invoke({
    "user_type": "owner",
    "query": "What is the recommended oil for my BMW?",
    "intent": "",
    "retrieved_chunks": [],
    "confidence_score": 0.0,
    "conversation_history": [],
    "citations": [],
    "image_paths": []
})

print(result["conversation_history"][-1].content)
print(result["confidence_score"])
print(result["citations"])


In [ ]:
# project folder architecture check :

import os
for root, dirs, files in os.walk("."):
    # skip hidden and cache folders
    dirs[:] = [d for d in dirs if d not in ['.git', '__pycache__', '.ipynb_checkpoints']]
    level = root.replace(".", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{root}/")
    for file in files:
        print(f"{indent}  {file}")

In [ ]:
# !pip freeze > requirements.txt

In [ ]:
import nbformat

with open("project_coding.ipynb", "r") as f:
    nb = nbformat.read(f, as_version=4)

for i, cell in enumerate(nb.cells):
    if cell.cell_type == "code":
        first_line = cell.source.split("\n")[0] if cell.source else "(empty)"
        print(f"Cell {i}: {first_line}")

# =======================================================

# Pending todos

# Gaurdrails

In [ ]:
INJECTION_PATTERNS = [
    "ignore previous instructions",
    "ignore all instructions",
    "you are now",
    "forget your instructions",
    "disregard your instructions"]  

def input_guardrail(state):
    query = state["query"]
      
    
    if len(query.strip()) == 0:
        return {
            "guardrail_status": "blocked_input",
            "guardrail_response": "Please Enter a valid input"
        }
    
    elif len(query.strip()) > 500:
        return {
            "guardrail_status": "blocked_input",
            "guardrail_response": "Query is too long. Please keep it under 500 characters."
        }
    
    elif any(pattern in query.lower() for pattern in INJECTION_PATTERNS):
        return {
            "guardrail_status": "blocked_input",
            "guardrail_response": "Invalid input detected."
        }
    
    elif profanity.contains_profanity(query):
        return {
                "guardrail_status": "blocked_input",
                "guardrail_response": "Only to answer sevice related questions"
            }  
    else:
        return {"guardrail_status": "pass", "guardrail_response": ""}

In [ ]:
def output_guardrail (state):
    response = state["conversation_history"][-1].content
    
    if any(pattern in response.lower() for pattern in INJECTION_PATTERNS):
        return {
            "guardrail_status": "blocked_output",
            "guardrail_response": "Invalid input detected.",
            "conversation_history": state["conversation_history"][:-1]
        }    
      
    elif profanity.contains_profanity(response):
        return {
                "guardrail_status": "blocked_output",
                "guardrail_response": "I'm unable to provide that response. Please consult a certified BMW technician.",
                "conversation_history": state["conversation_history"][:-1]  # remove last AIMessage
            }
    else:
        return {"guardrail_status": "pass", "guardrail_response": ""}

##### Graph - v2

In [ ]:
graph = StateGraph(AgentState)
graph.add_node('classifier', classifier)
graph.add_node('input_guardrail', input_guardrail)
graph.add_node('text_retriever', text_retriever)   
graph.add_node('unknown_handler', unknown_handler)   
graph.add_node("confidence", confidence_score)
graph.add_node("conversation", conversation)
graph.add_node('output_guardrail', output_guardrail)

graph.set_entry_point('input_guardrail')
graph.add_conditional_edges(
    "input_guardrail",      # after this node
    route_after_input_guard,      # call this function
    {
        "blocked_input": END,
        "pass": "classifier"
    }
)

graph.add_conditional_edges(
    "classifier",      # after this node
    route_intent,      # call this function
    {
        "text_retriever": "text_retriever",
       
        "unknown_handler" : "unknown_handler",
       
    }
)
graph.add_edge('text_retriever', 'confidence')   
graph.add_edge('unknown_handler', END)
graph.add_edge('confidence', 'conversation') 
graph.add_edge('conversation', "output_guardrail") 
graph.add_edge('output_guardrail', END) 


app = graph.compile()

In [ ]:
def route_after_input_guard(state):
    if state["guardrail_status"] == "pass":
        return "classifier"
    return "blocked_input"

##### Graph - v3 (final)

In [ ]:
from langgraph.graph import StateGraph, END
from agent.state import AgentState
from agent.classifier import classifier
from agent.retrievers import text_retriever
from agent.unknown_handler import unknown_handler
from agent.confidence_score import confidence_score
from agent.conversation import conversation
from agent.input_guardrail import input_guardrail
from agent.output_guardrail import output_guardrail

def route_after_input_guard(state):
    if state["guardrail_status"] == "pass":
        return "classifier"
    return "blocked_input"

def route_intent(state):
    intent = state["intent"]
    if intent == "both":
        return "text_retriever"  # no real tables in this manual
    elif intent == "table":
        return "text_retriever"  # no real tables in this manual
    elif intent == "unknown":
        return "unknown_handler"
    else:
        return "text_retriever"
    

graph = StateGraph(AgentState)
graph.add_node('input_guardrail', input_guardrail)
graph.add_node('classifier', classifier)
graph.add_node('text_retriever', text_retriever)   
graph.add_node('unknown_handler', unknown_handler)   
graph.add_node("confidence", confidence_score)
graph.add_node("conversation", conversation)
graph.add_node('output_guardrail', output_guardrail)

graph.set_entry_point('input_guardrail')
graph.add_conditional_edges(
    "input_guardrail",      # after this node
    route_after_input_guard,      # call this function
    {
        "blocked_input": END,
        "pass": "classifier"
    }
)
graph.add_conditional_edges(
    "classifier",      # after this node
    route_intent,      # call this function
    {
        "text_retriever": "text_retriever",
        
        "unknown_handler" : "unknown_handler"
    }
)

graph.add_edge('text_retriever', 'confidence')   
graph.add_edge('unknown_handler', END)
graph.add_edge('confidence', 'conversation') 
graph.add_edge('conversation', 'output_guardrail')
graph.add_edge('output_guardrail', END)

app = graph.compile()

# adding ingestion (pdf-extraction update/rebuild, chunker, vectorDB)

#### cheking the structure of pages to extract nitly - header, sub section, page number, sub header

In [ ]:
DATA_FOLDER = "C:/Users/Pranali Jadhav/OneDrive/Documents/GEN_AI/my_study/Bot_Project_1/graphprod/"

with pdfplumber.open(f"{DATA_FOLDER}/bmw_manual.pdf") as pdf:
    for page in pdf.pages[458:462]:  # first 10 pages
        print(f"\n--- PAGE {page.page_number} ---")
        # print(page.extract_text())
        print(page.extract_text()[:200]) # first 200 chars only
        print("---END---")

In [ ]:

DATA_FOLDER = "C:/Users/Pranali Jadhav/OneDrive/Documents/GEN_AI/my_study/Bot_Project_1/graphprod/"

import re

def clean_text(text):
    if not text:
        return None
    # remove footer
    text = re.sub(r'Online Edition for Part no\..*?II/\d+', '', text)
    # remove Seite X
    text = re.sub(r'Seite \d+', '', text)
    # before whitespace normalization
    text = re.sub(r'-\n', '', text)
    # remove standalone page numbers
    text = re.sub(r'^\d+$', '', text, flags=re.MULTILINE)
    # normalize whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def loader_doc():
    text_doc = []
    table_doc =[]
    image_doc = []

    for file in os.listdir(DATA_FOLDER):
        if file.endswith(".pdf"):
            # file_name = file.replace(".pdf", "")
            with pdfplumber.open(f"{DATA_FOLDER}/{file}") as pdf:
                
                if os.path.exists("images"):
                    shutil.rmtree("images")                
                os.makedirs("images", exist_ok=True)
                for page in pdf.pages:
                    if page.page_number <= 21 or page.page_number >= 460:
                        continue  # skip these pages
                    texts = clean_text(page.extract_text())
                    if texts:
                            text_doc.append(Document(
                                page_content=texts,
                                    metadata={
                                        "page_number": page.page_number,
                                        "section": f"page_{page.page_number}",
                                        "chunk_type": "text",
                                        "source_file": file
                                        # "image_path": {image_path} # image only
                                        }
                                    ))
                    tables = page.extract_tables()
                    for i, table in enumerate(tables):
                        table_doc.append(Document(
                                page_content=str(table),
                                    metadata={
                                        "page_number": page.page_number,
                                        "section": f"page_{page.page_number}",
                                        "chunk_type": "table",
                                        "source_file": file
                                        # "image_path": {image_path} # image only
                                        }
                                    ))
                    images = [img for img in page.images if img["srcsize"][0] >= 100 and img["srcsize"][1] >= 100]                        
                    for i, img in enumerate(images):
                            try:
                                raw_bytes = img["stream"].get_data()
                                pil_image = Image.open(BytesIO(raw_bytes))
                                if pil_image.mode == "CMYK":
                                    pil_image = pil_image.convert("RGB")
                                image_path = f"images/page_{page.page_number}_image_{i}.png"
                                pil_image.save(image_path)
                                image_doc.append(Document(
                                    page_content=f"images/page_{page.page_number}_image_{i}.png",
                                    metadata={
                                        "page_number": page.page_number,
                                        "section": f"page_{page.page_number}",
                                        "chunk_type": "image_description",
                                        "source_file": file,
                                        "image_path": image_path # image for only
                                        }
                                    ))
                            except Exception as e:
                                print(f"Skipped image on page {page.page_number}: {e}")
                                continue

    print(len(text_doc))
    print(len(table_doc))
    print(len(image_doc))
    return text_doc, table_doc, image_doc
                        
    

In [ ]:
text_doc, table_doc, image_doc = loader_doc()

In [ ]:
print(text_doc[0].page_content)

In [ ]:
chunks = chunker(text_doc)
print(f"Total chunks: {len(chunks)}")
print("\n--- SAMPLE CHUNK ---")
print(chunks[0].page_content)
print("\n--- METADATA ---")
print(chunks[0].metadata)

In [ ]:
print(f"Total chunks: {len(chunks)}")

In [ ]:
import os
os.environ["DATA_FOLDER"] = "C:/Users/Pranali Jadhav/OneDrive/Documents/GEN_AI/my_study/Bot_Project_1/graphprod"

import hashlib
import sys
sys.path.append("C:/Users/Pranali Jadhav/OneDrive/Documents/GEN_AI/my_study/Bot_Project_1/graphprod/deploy")
from scripts.loader import loader_doc

text_doc, table_doc, image_doc = loader_doc()
print(f"Total images extracted: {len(image_doc)}")

# check duplicates
seen = set()
unique = 0
for img in image_doc:
    with open(img.page_content, "rb") as f:
        h = hashlib.md5(f.read()).hexdigest()
    if h not in seen:
        seen.add(h)
        unique += 1

print(f"Unique images: {unique}")
print(f"Duplicates to skip: {len(image_doc) - unique}")

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
import sys
import os
sys.path.append("C:/Users/Pranali Jadhav/OneDrive/Documents/GEN_AI/my_study/Bot_Project_1/graphprod/deploy")
os.environ["DATA_FOLDER"] = "C:/Users/Pranali Jadhav/OneDrive/Documents/GEN_AI/my_study/Bot_Project_1/graphprod"

from agent.conversation import count_tokens
from config.settings import TOKEN_LIMIT

# simulate long history
fake_history = []
for i in range(50):
    fake_history.append(HumanMessage(content="what is the recommended oil for my BMW engine and what viscosity grade should I use?"))
    fake_history.append(AIMessage(content="For your BMW petrol engine you should use BMW Longlife-12 FE or Longlife-17 FE+ with SAE 0W-20 viscosity grade as recommended by BMW for optimal engine performance and fuel efficiency."))

# check token count
from agent.conversation import count_tokens
history_text = " ".join([m.content for m in fake_history])
print(f"History tokens: {count_tokens(history_text)}")
print(f"TOKEN_LIMIT: {TOKEN_LIMIT}")
print(f"Over limit: {count_tokens(history_text) > TOKEN_LIMIT}")

#### QUERY_VARIATIONS logic

In [ ]:
from config.settings import LLM_MODEL, client, QUERY_VARIATIONS_LIMIT
def query_expansion(state):
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": f"""Generate exactly {QUERY_VARIATIONS_LIMIT} alternative phrasings of the user query.
            Return only the {QUERY_VARIATIONS_LIMIT} variations, one per line, no numbering, no extra text."""},
            {"role": "user", "content": state["query"]}
        ]
    )
    raw = response.choices[0].message.content.strip()
    variations = [v.strip() for v in raw.split("\n") if v.strip()]
    return {"query_variations": variations}

In [ ]:
import pdfplumber

with pdfplumber.open("bmw_manual.pdf") as pdf:
    page = pdf.pages[22]  # page from your screenshot
    chars = page.chars
    sizes = set((c["fontname"], round(c["size"])) for c in chars)
    for s in sorted(sizes, key=lambda x: x[1], reverse=True):
        print(s)

In [ ]:
with pdfplumber.open("bmw_manual.pdf") as pdf:
    page = pdf.pages[22]
    for char in page.chars:
        if round(char["size"]) >= 11:
            print(round(char["size"]), char["fontname"], char.get("stroking_color"), char.get("non_stroking_color"), repr(char["text"]))

In [ ]:
with pdfplumber.open("bmw_manual.pdf") as pdf:
    page = pdf.pages[22]
    print(page.lines)

In [ ]:
with pdfplumber.open("bmw_manual.pdf") as pdf:
    page = pdf.pages[22]
    
    teal_lines = [l for l in page.lines 
                  if l.get("stroking_color") == (1, 0.6, 0, 0) and l["height"] == 0.0]
    
    teal_chars = [c for c in page.chars 
                  if c.get("non_stroking_color") == (1, 0.6, 0, 0)]
    
    print("Teal lines (y0):", [round(l["top"]) for l in teal_lines])
    print("Teal text sample:", [''.join([c["text"] for c in teal_chars[:20]])])
    print("Teal char tops:", sorted(set(round(c["top"]) for c in teal_chars)))

In [3]:
import fitz
DATA_FOLDER = "C:/Users/Pranali Jadhav/OneDrive/Documents/GEN_AI/my_study/Bot_Project_1/bmw_manual.pdf"
doc = fitz.open(DATA_FOLDER)
for level, title, page in doc.get_toc():
    print(level, title, page)

1 NOTES 6
2 Notes 6
3 About this Owner's Handbook 6
3 Media overview 6
3 Additional sources of information 7
3 Icons and displays 7
3 Vehicle equipment 8
3 Production date 8
3 Status of the Owner's Handbook 8
3 Your own safety 9
3 Vehicle data and data protection 10
3 Event data recorder 19
3 Vehicle identification number 20
1 QUICK REFERENCE 22
2 Getting in 22
3 Opening and closing 22
3 Displays, operating elements 23
2 Adjustment and operation 25
3 Seats, mirrors and steering wheel 25
3 Infotainment 26
2 On the move 28
3 Driving 28
3 Light and vision 29
3 Air conditioning 31
3 Pit stop 33
3 How to get assistance 34
1 CONTROLS 36
2 Vehicle cockpit 36
3 Vehicle equipment 36
3 Around the steering wheel 36
3 Around the centre console 38
3 Around the headliner 40
2 Sensors in the vehicle 41
3 Vehicle equipment 41
3 Overview 41
3 Cameras 41
3 Radar sensors 42
3 Ultrasonic sensors 44
2 Vehicle operating condition 46
3 Vehicle equipment 46
3 General 46
3 Rest state 46
3 Standby state 47
3 Dr

In [4]:
import fitz
doc = fitz.open(DATA_FOLDER)
page = doc[35]  # page 36, 0-indexed

# 1. Check image bbox(es) on the page
for img in page.get_images(full=True):
    xref = img[0]
    bbox = page.get_image_bbox(img)
    print("IMAGE bbox:", bbox, "page width:", page.rect.width)

# 2. Check raw text blocks — is the numbered legend text even extracted?
blocks = page.get_text("dict")["blocks"]
for b in blocks:
    if b["type"] == 0:  # text block
        text = "".join(span["text"] for line in b["lines"] for span in line["spans"])
        print("TEXT block bbox:", b["bbox"], "->", repr(text[:80]))

IMAGE bbox: Rect(14.173233032226562, 191.24371337890625, 390.25323486328125, 398.84368896484375) page width: 419.5275573730469
IMAGE bbox: Rect(26.929141998291016, 405.4443664550781, 50.929141998291016, 429.4443664550781) page width: 419.5275573730469
IMAGE bbox: Rect(26.929141998291016, 439.0450744628906, 50.929141998291016, 463.0450744628906) page width: 419.5275573730469
IMAGE bbox: Rect(222.5196990966797, 405.4443359375, 246.5196990966797, 429.4443359375) page width: 419.5275573730469
IMAGE bbox: Rect(222.5196990966797, 436.0450439453125, 246.5196990966797, 460.0450439453125) page width: 419.5275573730469
TEXT block bbox: (14.173230171203613, 39.20182800292969, 141.72023010253906, 64.94683074951172) -> 'Vehicle cockpit'
TEXT block bbox: (14.173230171203613, 81.18932342529297, 131.99725341796875, 100.1593246459961) -> 'Vehicle equipment'
TEXT block bbox: (14.1702299118042, 110.34722900390625, 191.01271057128906, 143.86473083496094) -> 'This chapter describes equipment, systemsand fu